# Module 2 — Product Analyst: Funnel & Retention

Covers the purchase funnel (spec §7.1) and the cohort repeat-purchase analysis (spec §7.2). The A/B test simulation (§7.3) lives in its own notebook, `ab_test_simulation.ipynb`.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "processed" / "olist.db"
conn = sqlite3.connect(DB_PATH)

def q(sql):
    return pd.read_sql(sql, conn)


## 1. Purchase funnel — overall drop-off

In [2]:
summary = q("SELECT * FROM v_funnel_summary")
summary.T

,0
n_placed,99441.00
n_approved,99281.00
n_shipped,97658.00
n_delivered,96476.00
n_reviewed,98673.00
pct_approved,99.84
pct_shipped_of_approved,98.37
pct_delivered_of_shipped,98.79
pct_reviewed_of_delivered,99.33
pct_reviewed_of_placed,99.23


In [3]:
stage_counts = summary[["n_placed", "n_approved", "n_shipped", "n_delivered", "n_reviewed"]].iloc[0]
fig = px.funnel(
    x=stage_counts.values,
    y=["Placed", "Approved", "Shipped", "Delivered", "Reviewed"],
    title="Overall purchase funnel",
)
fig.show()
# Whichever consecutive pair has the biggest count drop is the bottleneck stage —
# read it off stage_counts directly rather than assuming from the pct_ columns,
# since pct_shipped_of_approved etc. are relative to the PREVIOUS stage, not to n_placed.

## 2. Funnel segmented by category and region

Is the bottleneck universal, or concentrated?

In [4]:
stages = q("SELECT * FROM v_order_funnel_stages")

by_category = stages.groupby("category").agg(
    n=("placed", "sum"),
    pct_delivered=("delivered", "mean"),
    pct_reviewed=("reviewed", "mean"),
).query("n >= 100").sort_values("pct_delivered")
by_category.head(10)  # categories with the worst delivered-rate

,n,pct_delivered,pct_reviewed
category,,,
construction_tools_safety,160,0.962500,0.993750
fashion_male_clothing,110,0.963636,0.990909
consoles_games,1049,0.965682,0.990467
fashion_underwear_beach,121,0.966942,0.991736
unknown,1423,0.968377,0.992270
drinks,291,0.969072,0.996564
art,199,0.969849,0.989950
agro_industry_and_commerce,182,0.972527,1.000000
computers,181,0.977901,0.983425


In [5]:
by_region = stages.groupby("customer_state").agg(
    n=("placed", "sum"),
    pct_delivered=("delivered", "mean"),
).query("n >= 100").sort_values("pct_delivered")
by_region.head(10)  # states with the worst delivered-rate

,n,pct_delivered
customer_state,,
se,350,0.957143
ce,1336,0.957335
ma,747,0.959839
ro,253,0.960474
rj,12852,0.961173
al,413,0.961259
pi,495,0.961616
ba,3380,0.963314
pe,1652,0.964286


**Insight (fill in):** the biggest single-stage bottleneck is ___, and it is [universal across categories/regions | concentrated in ___].

## 3. Cohort repeat-purchase analysis

Cohort = month of each person's **first** purchase, keyed by `customer_unique_id` (see `DECISIONS.md` for why not `customer_id`). Since ~97% of customers in this dataset never order again, we report *repeat-purchase rate per cohort* rather than forcing a month-by-month retention curve onto data that doesn't support one.

In [6]:
cohort_rates = q("SELECT * FROM v_cohort_repeat_rate")
fig = px.bar(cohort_rates, x="cohort_month", y="repeat_rate_pct",
             title="Repeat-purchase rate by first-purchase cohort month")
fig.show()

### What predicts a second purchase?

In [7]:
cohort = q("SELECT * FROM v_customer_cohort")

# By first-order category (top 15 by volume, to keep this readable)
top_cats = cohort["first_order_category"].value_counts().head(15).index
by_cat = (
    cohort[cohort["first_order_category"].isin(top_cats)]
    .groupby("first_order_category")["is_repeat_purchaser"]
    .mean()
    .sort_values(ascending=False)
)
by_cat

first_order_category
furniture_decor          0.046875
bed_bath_table           0.044756
sports_leisure           0.039438
computers_accessories    0.029774
garden_tools             0.029368
health_beauty            0.028000
perfumery                0.027924
housewares               0.026621
toys                     0.023822
telephony                0.023815
baby                     0.022959
watches_gifts            0.021014
auto                     0.020468
cool_stuff               0.018141
electronics              0.016506
Name: is_repeat_purchaser, dtype: float64

In [8]:
# By first-order value bucket
cohort["value_bucket"] = pd.qcut(cohort["first_order_value"], 5, duplicates="drop")
cohort.groupby("value_bucket", observed=True)["is_repeat_purchaser"].mean()

value_bucket
(0.849, 39.0]        0.032630
(39.0, 68.0]         0.031535
(68.0, 107.5]        0.033065
(107.5, 176.99]      0.026070
(176.99, 13440.0]    0.028683
Name: is_repeat_purchaser, dtype: float64

In [9]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model_df = cohort.dropna(subset=["first_order_value", "first_order_state"]).copy()
model = smf.logit(
    "is_repeat_purchaser ~ first_order_value + C(first_order_state)",
    data=model_df,
).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.135846
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                            Logit Regression Results                           
===============================================================================
Dep. Variable:     is_repeat_purchaser   No. Observations:                94982
Model:                           Logit   Df Residuals:                    94954
Method:                            MLE   Df Model:                           27
Date:                 Tue, 15 Sep 2026   Pseudo R-squ.:                0.001961
Time:                         09:49:08   Log-Likelihood:                -12903.
converged:                        True   LL-Null:                       -12928.
Covariance Type:             nonrobust   LLR p-value:                  0.003773
==============================================================================================
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -2.8137      0.514     -5.473      0.000      -3.821      -1.806
C(first_order_state)[T.al]    -0.5728      0.591     -0.969      0.333      -1.732       0.586
C(first_order_state)[T.am]    -0.9544      0.777     -1.228      0.220      -2.478       0.569
C(first_order_state)[T.ap]    -1.2884      1.131     -1.139      0.255      -3.505       0.928
C(first_order_state)[T.ba]    -0.6523      0.524     -1.244      0.214      -1.680       0.376
C(first_order_state)[T.ce]    -1.2184      0.559     -2.180      0.029      -2.314      -0.123
C(first_order_state)[T.df]    -0.5932      0.530     -1.120      0.263      -1.631       0.445
C(first_order_state)[T.es]    -0.6281      0.531     -1.183      0.237      -1.669       0.413
C(first_order_state)[T.go]    -0.5445      0.530     -1.028      0.304      -1.583       0.494
C(first_order_state)[T.ma]    -0.8873      0.573     -1.550      0.121      -2.009       0.235
C(first_order_state)[T.mg]    -0.6061      0.517     -1.173      0.241      -1.619       0.407
C(first_order_state)[T.ms]    -0.6747      0.564     -1.197      0.231      -1.780       0.430
C(first_order_state)[T.mt]    -0.4770      0.547     -0.872      0.383      -1.550       0.596
C(first_order_state)[T.pa]    -0.7935      0.555     -1.429      0.153      -1.882       0.295
C(first_order_state)[T.pb]    -0.7487      0.585     -1.279      0.201      -1.896       0.399
C(first_order_state)[T.pe]    -0.8560      0.540     -1.586      0.113      -1.914       0.202
C(first_order_state)[T.pi]    -0.9517      0.605     -1.573      0.116      -2.137       0.234
C(first_order_state)[T.pr]    -0.6127      0.521     -1.177      0.239      -1.633       0.408
C(first_order_state)[T.rj]    -0.4764      0.516     -0.923      0.356      -1.488       0.535
C(first_order_state)[T.rn]    -0.8422      0.597     -1.410      0.159      -2.013       0.329
C(first_order_state)[T.ro]    -0.3163      0.616     -0.513      0.608      -1.524       0.891
C(first_order_state)[T.rr]    -0.8708      1.135     -0.768      0.443      -3.095       1.353
C(first_order_state)[T.rs]    -0.5608      0.520     -1.079      0.281      -1.580       0.458
C(first_order_state)[T.sc]    -0.7215      0.524     -1.376      0.169      -1.749       0.306
C(first_order_state)[T.se]    -0.8232      0.626     -1.315      0.189      -2.050       0.404
C(first_order_state)[T.sp]    -0.5587      0.515     -1.086      0.278      -1.567       0.450
C(first_order_state)[T.to]    -0.7374      0.641     -1.151      0.250      -1.993       0.518
first_order_value             -0.0005      0.000     -3.949      0.000      -0.001      -0.000
==============================================================================================
"""

**Insight (fill in):** the strongest predictor of a second purchase is ___. Practical read: [is this actionable, e.g. "first-order value doesn't matter much, but customers in region X repeat at 2x the rate"].